# COMI on RunPod ? B300 / Blackwell update
Upload this notebook and comi-runpod.zip to /workspace. Run cells in order with a Python 3 kernel (3.10?3.12).
This version upgrades existing repo launch/setup files automatically. It uses PyTorch 2.8.0/CUDA 12.9, disables hf_transfer, checks real GPU kernels, guards free disk space, records peak VRAM, and supports checkpoint resume after Pod restart.
Training stays full encoder + full fusion + LoRA decoder. Defaults remain segment_size=4096 and max_steps=10000 (about 0.884 epoch). The GPU name and available memory are read from the Pod; a MIG partition does not expose the full physical GPU.
No model/checkpoint cleanup is performed automatically.


## 1. Paths and training settings

In [ ]:
from pathlib import Path
import os
import json
import subprocess
import shutil
import zipfile
from collections import deque

REPO = Path('/workspace/comi')
ARCHIVE = Path('/workspace/comi-runpod.zip')
DATA = Path('/workspace/comi-data/hotpotqa')
OUTPUT = Path('/workspace/comi-output/hotpotqa')
SMOKE_OUTPUT = Path('/workspace/comi-output/hotpotqa-smoke')
LOG_DIR = Path('/workspace/comi-logs')
VENV = Path('/workspace/comi-venv')

MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
NUM_GPUS = 1
SEGMENT_SIZE = 4096
MERGE_SIZE = 16
MAX_STEPS = 10000  # About 0.884 epoch at effective batch 8; 11306 steps is about one epoch.
GRADIENT_ACCUMULATION_STEPS = 8

import fcntl
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOCK_FILE = LOG_DIR / 'hotpotqa-train.lock'
STATUS_FILE = LOG_DIR / 'hotpotqa-train-status.json'
LOG = LOG_DIR / 'hotpotqa-train.log'

def training_is_running():
    with LOCK_FILE.open('a') as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            return True
        fcntl.flock(lock, fcntl.LOCK_UN)
        return False

assert not training_is_running(), 'A training job is active. Monitor it before running setup again.'

if not (REPO / 'scripts/runpod_train.sh').exists():
    assert ARCHIVE.is_file(), f'Upload ZIP to {ARCHIVE}'
    REPO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as archive:
        for entry in archive.infolist():
            target = (REPO / entry.filename).resolve()
            assert target.is_relative_to(REPO.resolve()), 'Invalid archive path'
        archive.extractall(REPO)
print('Repository:', REPO)
subprocess.run(['nvidia-smi'], check=True)

# Apply the Blackwell update to existing repositories too.
UPDATED_FILES = {'scripts/runpod_setup.sh': '#!/usr/bin/env bash\nset -euo pipefail\ncd "$(dirname "${BASH_SOURCE[0]}")/.."\npython3 -m venv /workspace/comi-venv\nsource /workspace/comi-venv/bin/activate\npython -m pip install --upgrade pip\npython -m pip install --upgrade --no-cache-dir torch==2.8.0 --index-url https://download.pytorch.org/whl/cu129\npython -m pip install -r requirements-runpod.txt\npython -m pip check\npython -m pip freeze > /workspace/comi-environment.txt\nexport HF_HUB_ENABLE_HF_TRANSFER=0\npython scripts/check_cuda.py\necho \'Setup complete. Run: source /workspace/comi-venv/bin/activate\'\n', 'scripts/check_cuda.py': '"""Exercise BF16 CUDA kernels before spending time loading model weights."""\nimport argparse\n\n\ndef check_cuda(num_gpus=1):\n    import torch\n    if num_gpus < 1 or torch.cuda.device_count() < num_gpus:\n        raise SystemExit("Requested GPU count is unavailable")\n    print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)\n    print("Compiled architectures:", torch.cuda.get_arch_list())\n    for index in range(num_gpus):\n        with torch.cuda.device(index):\n            properties = torch.cuda.get_device_properties(index)\n            print(index, properties.name, "capability", torch.cuda.get_device_capability(index),\n                  round(properties.total_memory / 2**30, 1), "GiB")\n            if not torch.cuda.is_bf16_supported():\n                raise SystemExit(f"GPU {index} does not support BF16")\n            try:\n                x = torch.randn(64, 64, device=f"cuda:{index}",\n                                dtype=torch.bfloat16, requires_grad=True)\n                (x @ x).float().square().mean().backward()\n                # Exercise embedding and attention kernels used by the model too.\n                indices = torch.arange(8, device=x.device)\n                torch.nn.functional.embedding(indices, x)\n                q = x.detach().view(1, 1, 64, 64)\n                torch.nn.functional.scaled_dot_product_attention(q, q, q)\n                torch.cuda.synchronize(index)\n                del x, q, indices\n            except RuntimeError as error:\n                raise SystemExit(\n                    f"CUDA kernel check failed on GPU {index}: {error}\\n"\n                    "For Blackwell, install the prepared torch==2.8.0/cu129 stack "\n                    "and verify the Pod\'s NVIDIA driver is compatible."\n                ) from error\n            print("BF16 forward/backward, embedding and SDPA: OK")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--num-gpus", type=int, default=1)\n    check_cuda(parser.parse_args().num_gpus)\n', 'scripts/preflight.py': '"""Validate actual JSONL data and RunPod hardware before loading model weights."""\nimport argparse\nimport json\nimport shutil\nfrom pathlib import Path\n\n\ndef validate_file(path):\n    count = 0\n    with Path(path).open(encoding="utf-8") as handle:\n        for number, line in enumerate(handle, 1):\n            row = json.loads(line)\n            for key in ("input", "prompt"):\n                if not isinstance(row.get(key), str) or not row[key].strip():\n                    raise ValueError(f"{path}:{number}: invalid {key}")\n            answers = row.get("answer")\n            answers = [answers] if isinstance(answers, str) else answers\n            if not isinstance(answers, list) or not answers or not all(\n                isinstance(answer, str) and answer.strip() for answer in answers\n            ):\n                raise ValueError(f"{path}:{number}: invalid answer")\n            count += 1\n    if not count:\n        raise ValueError(f"Empty dataset: {path}")\n    return count\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train-file", required=True)\n    parser.add_argument("--eval-file", required=True)\n    parser.add_argument("--num-gpus", type=int, default=1)\n    parser.add_argument("--output-dir", required=True)\n    args = parser.parse_args()\n    for path in (args.train_file, args.eval_file):\n        print(path, validate_file(path), "valid rows")\n    import torch\n    import peft\n    import transformers\n    from check_cuda import check_cuda\n    check_cuda(args.num_gpus)\n    output = Path(args.output_dir)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    print("Free disk GiB:", round(shutil.disk_usage(output.parent).free / 2**30, 1))\n    print("Versions:", torch.__version__, transformers.__version__, peft.__version__)\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/runpod_train.sh': '#!/usr/bin/env bash\nset -euo pipefail\ncd "$(dirname "${BASH_SOURCE[0]}")/.."\nexport HF_HOME="${HF_HOME:-/workspace/hf-cache}"\nexport HF_HUB_ENABLE_HF_TRANSFER=0\nexport TOKENIZERS_PARALLELISM=false\nexport DATA_DIR="${DATA_DIR:-/workspace/comi-data/hotpotqa}"\nexport NUM_GPUS="${NUM_GPUS:-1}"\nexport MODEL_NAME="${MODEL_NAME:-Qwen/Qwen3-4B-Instruct-2507}"\nexport SEGMENT_SIZE="${SEGMENT_SIZE:-4096}"\nexport GRADIENT_ACCUMULATION_STEPS="${GRADIENT_ACCUMULATION_STEPS:-8}"\nexport OUTPUT_DIR="${OUTPUT_DIR:-/workspace/comi-output/hotpotqa}"\nexport TRAIN_FILE="${TRAIN_FILE:-${DATA_DIR}/train.jsonl}"\nexport TEST_FILE="${TEST_FILE:-${DATA_DIR}/eval.jsonl}"\nif [[ "${1:-}" == "smoke" ]]; then\n  shift\n  export TRAIN_FILE="${DATA_DIR}/smoke_train.jsonl"\n  export TEST_FILE="${DATA_DIR}/smoke_eval.jsonl"\n  export OUTPUT_DIR="${SMOKE_OUTPUT_DIR:-/workspace/comi-output/hotpotqa-smoke}"\n  export MAX_STEPS=3 SAVE_STEPS=3 EVAL_STEPS=3\n  export GRADIENT_ACCUMULATION_STEPS=1\nfi\npython scripts/preflight.py --train-file "$TRAIN_FILE" --eval-file "$TEST_FILE" --num-gpus "$NUM_GPUS" --output-dir "$OUTPUT_DIR"\nbash train.sh "$@"\n', 'scripts/runpod_eval.sh': '#!/usr/bin/env bash\nset -euo pipefail\ncd "$(dirname "${BASH_SOURCE[0]}")/.."\nexport HF_HOME="${HF_HOME:-/workspace/hf-cache}"\nexport HF_HUB_ENABLE_HF_TRANSFER=0\nexport MODEL_NAME="${MODEL_NAME:-Qwen/Qwen3-4B-Instruct-2507}"\nexport NUM_GPUS="${NUM_GPUS:-1}"\nexport TEST_FILE="${TEST_FILE:-/workspace/comi-data/hotpotqa/validation.jsonl}"\nexport OUTPUT_DIR="${OUTPUT_DIR:-/workspace/comi-output/hotpotqa}"\nexport RESTORE_FROM="${RESTORE_FROM:-${OUTPUT_DIR}}"\nexport NUM_SAMPLES="${NUM_SAMPLES:-0}"\nexport SEGMENT_SIZE="${SEGMENT_SIZE:-4096}"\nbash test.sh --max_new_tokens 128 "$@"\n', 'requirements-runpod.txt': '# Install CUDA PyTorch separately using scripts/runpod_setup.sh.\ntransformers==4.57.5\ndatasets==4.3.0\npeft==0.17.1\naccelerate==1.11.0\nsafetensors==0.7.0\nsentencepiece==0.2.1\npytest==8.4.2\n', 'scripts/notebook_job.py': "import fcntl\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nlock_path, status_path = map(Path, sys.argv[1:3])\nwith lock_path.open('a') as lock:\n    try:\n        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n    except BlockingIOError:\n        raise SystemExit('Another notebook training job is already running')\n    state = {'pid': os.getpid(), 'started_at': time.time(), 'command': sys.argv[3:], 'exit_code': None}\n    def save_status():\n        temporary = status_path.with_suffix('.json.tmp')\n        temporary.write_text(json.dumps(state, indent=2))\n        temporary.replace(status_path)\n    save_status()\n    try:\n        state['exit_code'] = subprocess.run(sys.argv[3:]).returncode\n    except BaseException as error:\n        state['error'] = repr(error)\n        state['exit_code'] = 1\n        raise\n    finally:\n        state['finished_at'] = time.time()\n        save_status()\n    sys.exit(state['exit_code'])\n"}
for name, source in UPDATED_FILES.items():
    target = REPO / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8', newline='\n')
print('Applied Blackwell setup/launcher updates')

# Record peaks inside the training process, including optimizer/eval/save phases.
training_utils = REPO / 'training_utils.py'
text = training_utils.read_text()
marker = '# B300 notebook: save CUDA peak memory'
if marker not in text:
    anchor = '        safetensors.torch.save_file(deduped, os.path.join(output_dir, "model.safetensors"))'
    assert text.count(anchor) == 1, 'Unexpected training_utils.py; inspect before patching'
    patch = anchor + "\n        " + marker + "\n" + """        if torch.cuda.is_available():
            import json as memory_json
            memory_stats = {
                'peak_allocated_GiB': torch.cuda.max_memory_allocated() / 2**30,
                'peak_reserved_GiB': torch.cuda.max_memory_reserved() / 2**30,
                'total_GiB': torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory / 2**30,
            }
            with open(os.path.join(output_dir, 'vram_peak.json'), 'w') as handle:
                memory_json.dump(memory_stats, handle, indent=2)
            print('CUDA memory peaks:', memory_stats)
"""
    backup = training_utils.with_suffix('.py.before-b300')
    if not backup.exists():
        backup.write_text(text)
    training_utils.write_text(text.replace(anchor, patch))

# Display free space on both model cache and output filesystems.
for location in [Path('/workspace'), OUTPUT.parent]:
    location.mkdir(parents=True, exist_ok=True)
    print(location, 'free GiB:', round(shutil.disk_usage(location).free / 2**30, 1))


## 2. Install/upgrade and check Blackwell kernels
Safe to re-run after restart; pip reuses matching packages. Downloads avoid pip cache duplication. The training venv stays separate from the notebook kernel.


In [ ]:
assert not training_is_running(), 'Do not upgrade the venv during training'
subprocess.run(['bash', 'scripts/runpod_setup.sh'], cwd=REPO, check=True)
PYTHON = VENV / 'bin/python'
ENV = os.environ.copy()
ENV.update({
    'PATH': str(VENV / 'bin') + os.pathsep + ENV['PATH'],
    'HF_HOME': '/workspace/hf-cache',
    'HF_HUB_ENABLE_HF_TRANSFER': '0',
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTHONUNBUFFERED': '1',
    'DATA_DIR': str(DATA),
    'TRAIN_FILE': str(DATA / 'train.jsonl'),
    'TEST_FILE': str(DATA / 'eval.jsonl'),
    'MODEL_NAME': MODEL_NAME,
    'NUM_GPUS': str(NUM_GPUS),
    'SEGMENT_SIZE': str(SEGMENT_SIZE),
    'MERGE_SIZE': str(MERGE_SIZE),
    'MAX_STEPS': str(MAX_STEPS),
    'GRADIENT_ACCUMULATION_STEPS': str(GRADIENT_ACCUMULATION_STEPS),
    'OUTPUT_DIR': str(OUTPUT),
    'SMOKE_OUTPUT_DIR': str(SMOKE_OUTPUT),
})
subprocess.run([str(PYTHON), '-m', 'pytest', '-q'], cwd=REPO, env=ENV, check=True)


## 3. Copy prepared data and verify its hashes
No dataset download is needed when using the provided ZIP. Existing data is verified instead of overwritten.

In [ ]:
import hashlib
if not DATA.exists():
    shutil.copytree(REPO / 'data/hotpotqa', DATA)
manifest = json.loads((DATA / 'manifest.json').read_text())
for name, info in manifest['files'].items():
    digest = hashlib.sha256()
    with (DATA / name).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    assert digest.hexdigest() == info['sha256'], f'Checksum mismatch: {name}'
    print(name, info['rows'], 'rows: OK')
subprocess.run([str(PYTHON), 'scripts/preflight.py',
                '--train-file', str(DATA / 'train.jsonl'),
                '--eval-file', str(DATA / 'eval.jsonl'),
                '--num-gpus', str(NUM_GPUS), '--output-dir', str(OUTPUT)],
               cwd=REPO, env=ENV, check=True)

def valid_checkpoint(path):
    required = ['model.safetensors', 'comi_model_args.json', 'trainer_state.json', 'optimizer.pt', 'scheduler.pt']
    if not all((path / name).is_file() for name in required):
        return False
    probe = "from safetensors import safe_open; import sys; f=safe_open(sys.argv[1],framework='pt',device='cpu'); assert len(list(f.keys())) > 0"
    result = subprocess.run([str(PYTHON), '-c', probe, str(path / 'model.safetensors')], env=ENV, capture_output=True, text=True)
    if result.returncode:
        print('Invalid checkpoint:', path, result.stderr[-500:])
        return False
    state = json.loads((path / 'trainer_state.json').read_text())
    return state.get('global_step', 0) > 0

SMOKE_CHECKPOINT = SMOKE_OUTPUT / 'checkpoint-3'
print('Existing smoke checkpoint valid:', valid_checkpoint(SMOKE_CHECKPOINT))
print('Full training checkpoints:', sorted(p.name for p in OUTPUT.glob('checkpoint-*')))

def require_disk(gib, location=None):
    location = Path(location) if location is not None else OUTPUT.parent
    location.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(location).free / 2**30
    print(f'Free output disk: {free:.1f} GiB; required planning reserve: {gib} GiB')
    if free < gib:
        raise RuntimeError('Increase /workspace storage or choose a larger OUTPUT/SMOKE_OUTPUT volume before training. No files were deleted.')


## 4. Smoke training or reuse an existing complete checkpoint
Checks free disk before training. A valid checkpoint-3 from the previous run is reused even if its final root save failed. If partial output exists without a valid checkpoint, select a new SMOKE_OUTPUT path in cell 1; old files are retained.


In [ ]:
assert not training_is_running(), 'Full training is active'
if valid_checkpoint(SMOKE_CHECKPOINT):
    print('Reusing existing smoke checkpoint:', SMOKE_CHECKPOINT)
else:
    if SMOKE_OUTPUT.exists() and any(SMOKE_OUTPUT.iterdir()):
        raise RuntimeError('Incomplete smoke output exists. Set a new SMOKE_OUTPUT in cell 1; existing files are retained.')
    require_disk(50, SMOKE_OUTPUT.parent)
    subprocess.run(['bash', 'scripts/runpod_train.sh', 'smoke', '--logging_steps', '1'], cwd=REPO, env=ENV, check=True)
    assert valid_checkpoint(SMOKE_CHECKPOINT)
for path in [SMOKE_CHECKPOINT / 'vram_peak.json', SMOKE_OUTPUT / 'vram_peak.json']:
    if path.exists():
        print(path, json.loads(path.read_text()))


## 5. Smoke inference

In [ ]:
smoke_env = ENV.copy()
smoke_env.update({'RESTORE_FROM': str(SMOKE_CHECKPOINT), 'OUTPUT_DIR': str(SMOKE_OUTPUT),
                  'TEST_FILE': str(DATA / 'smoke_eval.jsonl'), 'NUM_SAMPLES': '10'})
subprocess.run(['bash', 'scripts/runpod_eval.sh'], cwd=REPO, env=smoke_env, check=True)

SMOKE_INFERENCE_OK = True


## 6. Start/resume full training in the background
Start only after smoke inference succeeds. A file lock prevents duplicate jobs. After Pod restart, run cells 1?3 to rebuild notebook variables, reuse smoke checkpoint/inference, then run this cell to resume. Keep the original training configuration when resuming.
New runs require 100 GiB free; resumes require 40 GiB spare for the next checkpoint/final save. These are conservative planning guards, not exact disk predictions.


In [ ]:
if training_is_running():
    print('Training already running; inspect Monitor')
else:
    assert globals().get('SMOKE_INFERENCE_OK', False), 'Run smoke inference first'
    checkpoints = sorted(OUTPUT.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
    if checkpoints:
        latest = checkpoints[-1]
        assert valid_checkpoint(latest), f'Latest checkpoint is incomplete: {latest}; inspect it before resuming'
        print('Resume checkpoint:', latest)
    elif OUTPUT.exists() and any(OUTPUT.iterdir()):
        raise RuntimeError('Non-empty output has no resumable checkpoint. Choose a new OUTPUT path.')
    require_disk(40 if checkpoints else 100)
    with LOG.open('ab', buffering=0) as log_handle:
        process = subprocess.Popen(
            [str(PYTHON), 'scripts/notebook_job.py', str(LOCK_FILE), str(STATUS_FILE),
             'bash', 'scripts/runpod_train.sh', '--logging_steps', '10'],
            cwd=REPO, env=ENV, stdin=subprocess.DEVNULL,
            stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True,
        )
    print('Started training supervisor PID:', process.pid)
print('Log:', LOG)


## 7. Monitor
Re-run this cell to view progress. A missing process means inspect the log for successful completion or an error; it does not by itself mean success.

In [ ]:
print('Training process active:', training_is_running())
subprocess.run(['nvidia-smi'], check=True)
if LOG.exists():
    with LOG.open(encoding='utf-8', errors='replace') as handle:
        print(''.join(deque(handle, maxlen=60)))
print('Checkpoints:', sorted(path.name for path in OUTPUT.glob('checkpoint-*')))

if STATUS_FILE.exists():
    print('Job status (may describe a job before Pod restart):', json.loads(STATUS_FILE.read_text()))
peaks = list(OUTPUT.glob('checkpoint-*/vram_peak.json'))
if peaks:
    latest = max(peaks, key=lambda p: int(p.parent.name.split('-')[-1]))
    print('Latest recorded CUDA peak:', json.loads(latest.read_text()))


## 8. Evaluate after training finishes
Runs on all 7,405 validation examples. Results are answer EM/F1, excluding supporting-fact and joint HotpotQA scores.

In [ ]:
assert STATUS_FILE.exists() and json.loads(STATUS_FILE.read_text()).get('exit_code') == 0, 'Check training log/status before evaluation'
assert not training_is_running(), 'Wait for training to finish before evaluation'
assert (OUTPUT / 'model.safetensors').exists(), 'Final model is not available'
state = json.loads((OUTPUT / 'trainer_state.json').read_text())
assert state['global_step'] >= MAX_STEPS, 'Training did not reach the configured number of steps'
eval_env = ENV.copy()
eval_env.update({'RESTORE_FROM': str(OUTPUT), 'OUTPUT_DIR': str(OUTPUT),
                 'TEST_FILE': str(DATA / 'validation.jsonl'), 'NUM_SAMPLES': '0'})
subprocess.run(['bash', 'scripts/runpod_eval.sh'], cwd=REPO, env=eval_env, check=True)
for path in (OUTPUT / 'eval_result').glob('*metrics*.json'):
    print(path.name, json.loads(path.read_text()))


## 9. Save your results
Download `/workspace/comi-output/hotpotqa` and `/workspace/comi-logs` before deleting the Pod. Intermediate `checkpoint-N` directories include optimizer/RNG state for resume; the root contains the final model and COMI configuration.

If the GPU runs out of memory, do not switch training modes accidentally: this notebook retains the original full encoder/fusion and LoRA decoder configuration. Use a GPU with more VRAM, or separately configure optimizer offload and repeat the smoke test.
